In [ ]:
# Load and enhance data with sustainability metrics
import pandas as pd
import numpy as np
import pickle

# Load original data
df = pd.read_csv('../data/ai4i2020_local.csv')

def add_sustainability_metrics(df):
    """Add energy and emission metrics based on equipment degradation"""
    
    # Tool wear as proxy for degradation
    degradation = df['Tool wear [min]'] / df['Tool wear [min]'].max()
    
    # Energy consumption model
    base_energy = 85  # kWh (baseline for Nigerian industry)
    df['energy_kwh'] = base_energy * (1 + 0.2 * degradation) ** 1.15
    
    # Add realistic noise
    df['energy_kwh'] += np.random.normal(0, 3, len(df))
    
    # Nigerian grid emission factor: 0.47 kg CO2/kWh
    df['grid_co2_kg'] = df['energy_kwh'] * 0.47
    
    # Diesel generator usage (30% of time due to power cuts)
    df['on_diesel'] = np.random.binomial(1, 0.3, len(df))
    df['diesel_hours'] = df['on_diesel'] * np.random.uniform(2, 6, len(df))
    df['diesel_co2_kg'] = df['diesel_hours'] * 8.5  # kg CO2 per hour
    
    # Total emissions
    df['total_co2_kg'] = df['grid_co2_kg'] + df['diesel_co2_kg']
    
    # Maintenance urgency score (0-100)
    df['maintenance_urgency'] = (degradation * 70 + 
                                 df['Machine failure'] * 30)
    
    return df

# Apply sustainability metrics
df_enhanced = add_sustainability_metrics(df)

print("Sustainability metrics added:")
print(df_enhanced[['energy_kwh', 'total_co2_kg', 'diesel_hours', 
                   'maintenance_urgency']].describe())

In [ ]:
# Simulate power instability
def simulate_nigerian_power_conditions(df):
    """Simulate power grid issues common in Nigeria"""
    
    n = len(df)
    
    # Power conditions: stable, brownout, outage
    power_conditions = np.random.choice(
        ['stable', 'brownout', 'outage'],
        n,
        p=[0.65, 0.25, 0.10]  # Nigerian grid reality
    )
    
    df['power_condition'] = power_conditions
    
    # Affect sensor readings based on power
    for col in ['Rotational speed [rpm]', 'Torque [Nm]']:
        original = df[col].copy()
        
        # Brownout: 10-20% reduction
        brownout_mask = power_conditions == 'brownout'
        df.loc[brownout_mask, col] = original[brownout_mask] * np.random.uniform(0.8, 0.9)
        
        # Outage: equipment stops
        outage_mask = power_conditions == 'outage'
        df.loc[outage_mask, col] = 0
    
    return df

# Apply power simulation
df_enhanced = simulate_nigerian_power_conditions(df_enhanced)

print("Power conditions added:")
print(df_enhanced['power_condition'].value_counts())

In [ ]:
# Save enhanced dataset
df_enhanced.to_csv('../data/ai4i2020_enhanced2.csv', index=False)
print("Enhanced dataset saved to ../data/ai4i2020_enhanced.csv")

# Show sample
print("\nSample of enhanced data:")
print(df_enhanced[['Tool wear [min]', 'energy_kwh', 'total_co2_kg', 
                   'power_condition', 'maintenance_urgency']].head(10))